In [1]:
import json
import joblib
import numpy as np
import pandas as pd
import xgboost as xgb
import shap

In [6]:
MODEL_DIR = r"C:\glassIDS\models"
model = xgb.XGBClassifier()
model.load_model(rf"{MODEL_DIR}\xgboost_baseline.json")
le = joblib.load(rf"{MODEL_DIR}\label_encoder.joblib")
explainer = shap.TreeExplainer(model)

def class_contrib(sv, class_idx):
    # newer shap returns (rows, features, classes); older returns a list per class
    if isinstance(sv, np.ndarray) and sv.ndim == 3:
        return sv[0, :, class_idx]
    return sv[class_idx][0]

def build_alert_packet(x_row, alert_id, top_k=6):
    x = x_row.to_frame().T.astype(float)
    proba = model.predict_proba(x)[0]
    first, second = np.argsort(proba)[::-1][:2]
    contrib = class_contrib(explainer.shap_values(x), first)
    order = np.argsort(np.abs(contrib))[::-1][:top_k]
    return {
        "alert_id": alert_id,
        "predicted_class": le.classes_[first],
        "confidence": round(float(proba[first]), 4),
        "runner_up_class": le.classes_[second],
        "runner_up_confidence": round(float(proba[second]), 4),
        "top_features": [
            {"feature": x.columns[i],
             "value": round(float(x.iloc[0, i]), 4),
             "shap": round(float(contrib[i]), 4)}
            for i in order
        ],
    }


In [3]:
from sklearn.model_selection import train_test_split

CLEANED_PATH = r"C:\glassIDS\data\processed\cicids2017_cleaned.csv"
df = pd.read_csv(CLEANED_PATH)

feature_cols = [c for c in df.columns
                if c not in ("Label", "is_attack", "source_file", "Destination Port")]

X = df[feature_cols]
y = df["Label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print(X_train.shape, X_test.shape)   # expect (2016638, 77) (504160, 77)
assert list(X_test.columns) == model.get_booster().feature_names

(2016638, 77) (504160, 77)


In [4]:
pred_idx = model.predict(X_test)
pred_lbl = le.inverse_transform(pred_idx)
true_lbl = y_test.to_numpy()

# a clean attack: correctly predicted PortScan
clean_pos = np.where((true_lbl == "PortScan") & (pred_lbl == "PortScan"))[0][0]

# the shaky case: a real BENIGN flow the model called Bot
fp_pos = np.where((true_lbl == "BENIGN") & (pred_lbl == "Bot"))[0][0]

print(clean_pos, fp_pos, len(np.where((true_lbl == "BENIGN") & (pred_lbl == "Bot"))[0]))

1 5237 397


In [5]:
p_clean = build_alert_packet(X_test.iloc[clean_pos], alert_id=f"test-{clean_pos}")
p_fp    = build_alert_packet(X_test.iloc[fp_pos],    alert_id=f"test-{fp_pos}")

print(json.dumps(p_clean, indent=2))
print(json.dumps(p_fp, indent=2))

{
  "alert_id": "test-1",
  "predicted_class": "PortScan",
  "confidence": 0.9989,
  "runner_up_class": "BENIGN",
  "runner_up_confidence": 0.0011,
  "top_features": [
    {
      "feature": "Bwd Packets/s",
      "value": 125000.0,
      "shap": 3.6406
    },
    {
      "feature": "PSH Flag Count",
      "value": 1.0,
      "shap": 1.1446
    },
    {
      "feature": "Bwd Packet Length Min",
      "value": 6.0,
      "shap": 1.0006
    },
    {
      "feature": "Flow IAT Mean",
      "value": 8.0,
      "shap": 0.8357
    },
    {
      "feature": "Packet Length Mean",
      "value": 3.3333,
      "shap": 0.174
    },
    {
      "feature": "Init_Win_bytes_forward",
      "value": 1024.0,
      "shap": -0.1475
    }
  ]
}
{
  "alert_id": "test-5237",
  "predicted_class": "Bot",
  "confidence": 0.998,
  "runner_up_class": "BENIGN",
  "runner_up_confidence": 0.002,
  "top_features": [
    {
      "feature": "min_seg_size_forward",
      "value": 28.0,
      "shap": 1.8038
    },
    {